# Notebook 2: AE and VAE Training

This notebook trains the Autoencoder (AE) and the Variational Autoencoder (VAE) on both datasets.
The model code lives in `thesis_models.py` (same folder), so Notebook 3 uses exactly the same models.

**Identical conditions for both models**

| Setting | Value |
|---|---|
| Hidden layers (encoder, mirrored in decoder) | IEEE-CIS: 256, 128, 64 / BAF: 64, 48, 32 |
| Activation, dropout | ReLU, 0.2 after every hidden layer |
| Latent size | searched over 8, 16, 32 |
| Reconstruction loss | sum of squared errors per transaction (MSE), identical for AE and VAE, no focal loss |
| VAE extra | Gaussian latent (mean, log-variance), reparameterisation trick, closed-form KL term weighted by beta = 0.01 (beta-VAE) with a 10-epoch warm-up; log-variance starts at -6 |
| Optimiser | Adam, learning rate 0.001, batch size 2048, at most 150 epochs |
| Schedule | reduce learning rate by half after 5 epochs without improvement |
| Early stopping | patience 10 on the early-stop set (normals only); an epoch counts as better only if the loss falls by more than 0.1%; best weights restored |
| Anomaly score | mean squared reconstruction error per transaction (VAE uses the latent mean) |
| Seeds | 0 to 4, results reported as mean and standard deviation |

**Two stages**

* **Stage A (tuning):** every latent size is trained once (seed 0) for each model and dataset.
  The latent size is chosen per model by the validation AUC-PR. This uses validation labels, so it is
  label-assisted, exactly like the calibrated threshold. Test data is never used to choose anything.
* **Stage B (final runs):** the chosen latent size is trained with seeds 1 to 4 (seed 0 is reused from Stage A).

Every run is saved to `../Results/runs_final/`. If the notebook stops, just run it again: finished runs are skipped.

**Why beta = 0.01 and not the standard beta = 1**

A first round of runs with the standard VAE (beta = 1) suffered posterior collapse: only 0 of 16 latent
dimensions (BAF) and 1 of 16 (IEEE-CIS) were active, so the latent space carried almost no information and
the VAE behaved like a model that predicts the average transaction. Because the inputs are scaled to [0, 1],
the reconstruction errors are small, and the full KL penalty outweighs them. A smaller KL weight (beta-VAE)
and a small starting sampling noise keep the latent space in use. Those first runs are kept in
`../Results/runs_beta1/` and are reported as an ablation in the thesis.

**At a glance**

| | |
|---|---|
| Run on | Google Colab with a GPU (the final runs used Colab). It also runs on a laptop CPU, only much slower |
| Run time | Depends on the hardware. Finished runs are skipped, so the notebook can be restarted safely |
| Reads | `../Processed/*.npz`, `thesis_models.py` |
| Writes | `../Results/runs_final/<run>/` (scores, model weights, history, meta), `best_latent.json`, `tuning_seed0.csv`, `training_summary.csv`, `../Figures/training_curves.png` |
| Run order | Step 2 of 6, after Notebook 01. |

In [1]:
import os, sys, json, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Works both on the laptop and on Google Colab.
# On Colab, upload the 04_Final_Thesis folder (with Processed/ and Final_Code/thesis_models.py)
# to "My Drive" first. The code picks the GPU automatically when there is one (see thesis_models.DEVICE).
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/04_Final_Thesis"
    sys.path.insert(0, os.path.join(BASE, "Final_Code"))
else:
    BASE = os.path.abspath("..")
    sys.path.insert(0, os.getcwd())
import thesis_models as tm
print("Running on:", tm.DEVICE, "| Colab:", IN_COLAB)

PROC = os.path.join(BASE, "Processed")
RUNS = os.path.join(BASE, "Results", "runs_final")
FIG_DIR = os.path.join(BASE, "Figures")
os.makedirs(RUNS, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

HIDDEN  = {"ieee": (256, 128, 64), "baf": (64, 48, 32)}
LATENTS = [8, 16, 32]
SEEDS   = [0, 1, 2, 3, 4]
MODELS  = ["AE", "VAE"]
BETA    = 0.01          # KL weight of the VAE (see the note at the top)

def load(ds):
    d = np.load(os.path.join(PROC, f"{ds}.npz"))
    return {k: d[k] for k in ["X_train", "X_es", "X_val", "y_val", "X_test", "y_test"]}

print("torch", torch.__version__)

Mounted at /content/drive
Running on: cuda | Colab: True
torch 2.14.1+cu130


## Stage A: latent size search (seed 0)

12 runs in total (2 datasets x 2 models x 3 latent sizes). BAF runs first because it is faster.

In [ ]:
t_all = time.time()
for ds in ["baf", "ieee"]:
    data = load(ds)
    for kind in MODELS:
        for z in LATENTS:
            tm.run_experiment(ds, kind, data, HIDDEN[ds], z, seed=0, out_root=RUNS, beta=BETA)
    del data
print(f"Stage A finished in {(time.time()-t_all)/60:.0f} min")

baf_AE_z8_s0: training ...
  epoch   1 | train 4.5747 | es 3.4754 (rec 3.4754, kl 0.0000) | lr 1.0e-03 | 2s
  epoch   5 | train 2.5601 | es 2.1725 (rec 2.1725, kl 0.0000) | lr 1.0e-03 | 7s
  epoch  10 | train 2.2149 | es 1.8357 (rec 1.8357, kl 0.0000) | lr 1.0e-03 | 12s
  epoch  15 | train 2.0223 | es 1.6962 (rec 1.6962, kl 0.0000) | lr 1.0e-03 | 19s
  epoch  20 | train 1.9672 | es 1.6512 (rec 1.6512, kl 0.0000) | lr 1.0e-03 | 24s
  epoch  25 | train 1.9464 | es 1.6246 (rec 1.6246, kl 0.0000) | lr 1.0e-03 | 31s
  epoch  30 | train 1.9341 | es 1.6180 (rec 1.6180, kl 0.0000) | lr 1.0e-03 | 37s
  epoch  35 | train 1.9285 | es 1.6059 (rec 1.6059, kl 0.0000) | lr 1.0e-03 | 42s
  epoch  40 | train 1.9235 | es 1.6095 (rec 1.6095, kl 0.0000) | lr 1.0e-03 | 48s
  epoch  45 | train 1.9161 | es 1.5970 (rec 1.5970, kl 0.0000) | lr 1.0e-03 | 54s
  epoch  50 | train 1.9170 | es 1.5943 (rec 1.5943, kl 0.0000) | lr 1.0e-03 | 60s
  epoch  55 | train 1.8921 | es 1.5833 (rec 1.5833, kl 0.0000) | lr 5.0e-

In [ ]:
def collect():
    rows = []
    for name in sorted(os.listdir(RUNS)):
        p = os.path.join(RUNS, name, "meta.json")
        if os.path.exists(p):
            with open(p) as f:
                rows.append(json.load(f))
    return pd.DataFrame(rows)

res = collect()
tuning = res[res.seed == 0].pivot_table(index=["dataset", "model"], columns="latent",
                                        values=["val_auc_pr", "val_auc_roc"]).round(4)
tuning

val_auc_pr                 val_auc_roc                
latent                8       16      32          8       16      32
dataset model                                                       
baf     AE        0.0136  0.0132  0.0139      0.5050  0.4938  0.5194
        VAE       0.0153  0.0138  0.0138      0.5478  0.5099  0.5096
ieee    AE        0.1614  0.1636  0.1659      0.7101  0.7207  0.7221
        VAE       0.1499  0.1546  0.1541      0.7082  0.7144  0.7060

In [ ]:
# Best latent size per dataset and model, chosen by validation AUC-PR
best = (res[res.seed == 0].sort_values("val_auc_pr", ascending=False)
        .groupby(["dataset", "model"]).head(1)[["dataset", "model", "latent", "val_auc_pr", "val_auc_roc"]]
        .sort_values(["dataset", "model"]).reset_index(drop=True))
BEST_Z = {(r.dataset, r.model): int(r.latent) for r in best.itertuples()}
with open(os.path.join(BASE, "Results", "best_latent.json"), "w") as f:
    json.dump({f"{k[0]}_{k[1]}": v for k, v in BEST_Z.items()}, f, indent=2)
res[res.seed == 0].to_csv(os.path.join(BASE, "Results", "tuning_seed0.csv"), index=False)
best

,dataset,model,latent,val_auc_pr,val_auc_roc
0,baf,AE,32,0.013902,0.519401
1,baf,VAE,8,0.015304,0.547835
2,ieee,AE,32,0.165895,0.722075
3,ieee,VAE,16,0.154571,0.714379


## Stage B: final runs with seeds 1 to 4

8 configurations x 4 seeds = 16 runs. This is the longest part, so it can be left running
(for example overnight). If it stops, run this cell again and it continues where it left off.

In [ ]:
t_all = time.time()
for ds in ["baf", "ieee"]:
    data = load(ds)
    for kind in MODELS:
        for s in SEEDS[1:]:
            tm.run_experiment(ds, kind, data, HIDDEN[ds], BEST_Z[(ds, kind)], seed=s, out_root=RUNS, beta=BETA)
    del data
print(f"Stage B finished in {(time.time()-t_all)/60:.0f} min")

## Training summary and training curves

In [ ]:
res = collect()
final = pd.concat([res[(res.dataset == ds) & (res.model == m) & (res.latent == z)]
                   for (ds, m), z in BEST_Z.items()])
cols = ["epochs_run", "best_epoch", "train_seconds", "n_params", "val_auc_pr", "val_auc_roc"]
if "active_latent_units" in final:
    cols.append("active_latent_units")
summary = final.groupby(["dataset", "model", "latent"])[cols].agg(["mean", "std"]).round(3)
summary.to_csv(os.path.join(BASE, "Results", "training_summary.csv"))
summary

epochs_run         best_epoch         train_seconds  \
                           mean     std       mean     std          mean   
dataset model latent                                                       
baf     AE    32          103.6  18.770      102.0  19.824        117.94   
        VAE   8            92.6  22.546       90.8  23.317        136.68   
ieee    AE    32          150.0   0.000      148.4   1.949         88.18   
        VAE   16          137.4  21.961      135.6  21.007        104.68   

                              n_params      val_auc_pr        val_auc_roc  \
                         std      mean  std       mean    std        mean   
dataset model latent                                                        
baf     AE    32      21.331   18034.0  0.0      0.014  0.001       0.515   
        VAE   8       33.443   16738.0  0.0      0.014  0.001       0.520   
ieee    AE    32       0.427  301891.0  0.0      0.167  0.002       0.722   
        VAE   16      16.920  300867.0  0.0      0.155  0.003       0.714   

                            active_latent_units         
                        std                mean    std  
dataset model latent                                    
baf     AE    32      0.028                 NaN    NaN  
        VAE   8       0.039                 5.0  0.000  
ieee    AE    32      0.003                 NaN    NaN  
        VAE   16      0.003                 5.6  0.548

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, ((ds, m), z) in zip(axes.ravel(), sorted(BEST_Z.items())):
    run = f"{ds}_{m}_z{z}_s0" if m == "AE" else f"{ds}_{m}_b{BETA:g}_z{z}_s0"
    with open(os.path.join(RUNS, run, "history.json")) as f:
        h = json.load(f)
    ep = range(1, len(h["train_loss"]) + 1)
    ax.plot(ep, h["train_loss"], label="Training loss")
    ax.plot(ep, h["es_loss"], label="Early-stop loss (normals)")
    ax.set_title(f"{'IEEE-CIS' if ds == 'ieee' else 'BAF'}: {m} (latent {z})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "training_curves.png"), dpi=300, bbox_inches="tight")
plt.show()